In [ ]:
import os
import re
import cv2
import numpy as np
png_root = 'voxelmorph/data/image slice-T1'
target_sz = (224, 192) 

def numeric_key(fname):
    m = re.match(r'^(\d+)\.png$', fname)
    return int(m.group(1)) if m else -1

patient_dirs = sorted(
    os.path.join(png_root, d) for d in os.listdir(png_root)
    if os.path.isdir(os.path.join(png_root, d))
)
volumes = []
for pd in patient_dirs:
    fnames = sorted(
        (f for f in os.listdir(pd) if f.lower().endswith('.png')),
        key=numeric_key
    )
    slices = []
    for fname in fnames:
        path = os.path.join(pd, fname)
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None or img.max() == 0:
            continue
        img = cv2.resize(img, target_sz, interpolation=cv2.INTER_LINEAR)
        img = img.astype(np.float32) / 255.0
        slices.append(img)
    if len(slices) > 0:
        volumes.append(np.stack(slices, axis=-1))  

print(f"Loaded {len(volumes)} patient volumes.")

slice_idx = volumes[0].shape[-1] // 2

slices_across = []
for vol in volumes:
    if slice_idx < vol.shape[-1]:
        sl = vol[:, :, slice_idx]
        slices_across.append(sl)
slices_across = np.stack(slices_across, axis=0)[..., np.newaxis]  

print(f"Extracted slice {slice_idx} from each patient → {slices_across.shape}")

def make_inter_subject_pairs(x, num_pairs=1000):
    N = x.shape[0]
    moving, fixed = [], []
    for _ in range(num_pairs):
        i, j = np.random.choice(N, 2, replace=False)
        moving.append(x[i])
        fixed.append(x[j])
    return np.array(moving), np.array(fixed)

moving, fixed = make_inter_subject_pairs(slices_across, num_pairs=2000)
zeros = np.zeros_like(moving)

print(f"Built {moving.shape[0]} inter-subject training pairs.")

np.save("ixi_slice_across_patients.npy", slices_across)
np.savez("ixi_pairs.npz", moving=moving, fixed=fixed, zeros=zeros)

print("Saved:")
print("   • ixi_slice_across_patients.npy")
print("   • ixi_pairs.npz (moving, fixed, zeros)")



Loaded 481 patient volumes.
Extracted slice 25 from each patient → (481, 192, 224, 1)
Built 2000 inter-subject training pairs.
Saved:
   • ixi_slice_across_patients.npy
   • ixi_pairs.npz (moving, fixed, zeros)


In [ ]:
import numpy as np
import pandas as pd

data = np.load("ixi_pairs.npz")

summary = pd.DataFrame([
    {"array_key": key, "shape": data[key].shape}
    for key in data.files
])
print(summary)

  array_key                shape
0    moving  (2000, 192, 224, 1)
1     fixed  (2000, 192, 224, 1)
2     zeros  (2000, 192, 224, 1)
